# Bagian yang mengubah model dari sekadar "akurat" menjadi "bisa dipertanggungjawabkan" ke bisnis dan bisa dijelaskan ke nasabah.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

model_final = lgb.Booster(model_file=str(MODELS_DIR / "lgb_model_final.txt"))
y_test = np.load(PROJECT_ROOT / "data" / "processed" / "y_test.npy")
final_proba = np.load(PROJECT_ROOT / "data" / "processed" / "final_proba.npy")

# X_test perlu di-generate ulang dari parquet + split yang sama (random_state sama = hasil identik)
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "application_features.parquet")
from sklearn.model_selection import train_test_split
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]
_, X_test, _, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_cols = X_test.select_dtypes(include=["category"]).columns.tolist()
for col in cat_cols:
    X_test[col] = X_test[col].astype("category")

## 1. Analisis SHAP (±1.5 jam)

In [ ]:
import shap

explainer = shap.TreeExplainer(model_final)
shap_values = explainer.shap_values(X_test)

# Untuk LightGBM binary classification, shap_values bisa berupa array tunggal atau list [neg, pos]
if isinstance(shap_values, list):
    shap_values_plot = shap_values[1]
else:
    shap_values_plot = shap_values

### a. Global feature importance — fitur mana yang paling berpengaruh secara keseluruhan:

In [ ]:
import matplotlib.pyplot as plt

shap.summary_plot(shap_values_plot, X_test, plot_type="bar", show=False, max_display=20)
plt.tight_layout()
plt.savefig("../reports/shap_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### b. Summary plot (arah pengaruh) — tidak cuma seberapa penting, tapi apakah nilai tinggi fitur itu menaikkan atau menurunkan risiko:

In [ ]:
shap.summary_plot(shap_values_plot, X_test, show=False, max_display=20)
plt.tight_layout()
plt.savefig("../reports/shap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

### c. Penjelasan untuk satu nasabah individual — ini bagian yang sering dicari recruiter karena menunjukkan model bisa "dijelaskan", bukan cuma kotak hitam:

In [ ]:
# Ambil satu nasabah dengan risiko tertinggi sebagai contoh
idx_highest_risk = np.argmax(final_proba)

shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    shap_values_plot[idx_highest_risk],
    X_test.iloc[idx_highest_risk],
    matplotlib=True, show=False
)
plt.savefig("../reports/shap_individual_explanation.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Probabilitas default nasabah ini: {final_proba[idx_highest_risk]:.4f}")

## 2. Kalibrasi Probabilitas (±1 jam)

LightGBM/XGBoost kadang menghasilkan skor yang urutannya benar (ranking bagus, makanya AUC tinggi) tapi nilai probabilitasnya sendiri tidak akurat (misal model bilang 80% padahal realitanya cuma 50% dari yang diberi skor segitu benar-benar default). Ini penting diperiksa karena langkah 3 (threshold berbasis cost) butuh probabilitas yang terkalibrasi dengan baik, bukan cuma ranking yang benar.

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, final_proba, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", label="Model LightGBM")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Kalibrasi sempurna")
plt.xlabel("Probabilitas prediksi rata-rata")
plt.ylabel("Proporsi aktual default")
plt.title("Reliability Diagram")
plt.legend()
plt.savefig("../reports/calibration_curve.png", dpi=150)
plt.show()